# 01. Exploratory Data Analysis

Course: Data Mining Techniques (X_400111), VU Amsterdam, 2026.
Team: VU-DM-2026-Group-5.

Dataset: Expedia 2013 personalized search competition.

Verified facts used downstream:
- Train: 4,958,347 rows x 54 columns.
- Test: 4,959,183 rows x 50 columns.
- Train-only columns dropped before modelling: `position`, `click_bool`, `booking_bool`, `gross_bookings_usd`.
- Target rates on train: clicks 4.475 percent, bookings 2.791 percent.
- Search-list length: min 5, max 38, mean 24.82, median 29.
- IDCG=0 share: 0 percent (every `srch_id` has at least one click or booking).
- Date range train and test both 2012-11-01 to 2013-06-30.

## Imports and paths

In [ ]:
from pathlib import Path
import sys
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA_DIR = ROOT / 'data'
FIG_DIR = ROOT / 'report' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Load processed parquet

If you only have CSVs, run `scripts/00_csv_to_parquet.py` from the working folder, or convert here once.

In [ ]:
TRAIN_PARQUET = DATA_DIR / 'processed' / 'train_clean.parquet'
TEST_PARQUET = DATA_DIR / 'processed' / 'test_clean.parquet'
# train = pd.read_parquet(TRAIN_PARQUET)
# test  = pd.read_parquet(TEST_PARQUET)
# train.shape, test.shape

## Schema and column parity

Train has 54 columns, test has 50. The four train-only columns are `position`, `click_bool`, `booking_bool`, and `gross_bookings_usd`. They are dropped before the model sees the matrix; we never reconstruct them on test.

## Target and metric

Relevance is derived from interaction labels:
- 5 if `booking_bool` is 1
- 1 if `click_bool` is 1 and `booking_bool` is 0
- 0 otherwise

We optimise NDCG@5. Our local IDCG=0 convention is to return 0, but the share of IDCG=0 queries on train and on the locked holdout is 0 percent, so the convention does not affect any reported number.

## Search-list length

Lists per `srch_id` range from 5 to 38 properties (mean 24.82, median 29). The distribution is bimodal: a long flat low-frequency segment from length 5 to 25 (around 4{,}000 searches each) and a strong peak between 30 and 34 (peak around 30{,}000 at length 32). This motivates the choice of NDCG@5 as the truncation depth.

![list length](../report/figures/list_length.png)

## Price

`price_usd` is heavily long-tailed: median 122 USD, 99th percentile 599 USD, with a few extreme outliers extending into the millions. If a histogram is shown the x-axis must use a log transform; otherwise the bulk of the distribution collapses against the left edge.

![price log](../report/figures/price_distribution_log.png)

## Correlations with booking

Spearman rank correlations between the numeric features and `booking_bool` are uniformly small. The strongest absolute correlations on a 5 percent search-stratified sample are `random_bool` (-0.0896) and `prop_location_score2` (+0.0760); all other features sit below 0.04 in absolute value. This already tells us a single-feature heuristic cannot do well; the strongest single-feature heuristic on the locked holdout is `prop_location_score2` desc with NDCG@5 = 0.2836, well below the LightGBM baseline.

![spearman](../report/figures/spearman_corr_with_booking.png)

## Position and the random_bool subset

`position` exists only in train. We drop it, since reconstructing it on test would require a target-encoded estimator and that is leakage-prone. The `random_bool=1` rows are not perfectly uniform in position; both `random_bool=0` and `random_bool=1` lists decay with position, with very low counts at positions 5, 11, 17, 23, and partly 29 (a UI artefact). This is why we do not use the simple Bellucci uniformity assumption to build a position-bias correction.

## Heuristic baselines on the locked holdout

| heuristic | NDCG@5 |
|---|---|
| random ranking (3 seeds) | 0.1579 (std 0.0008) |
| `prop_starrating` desc | 0.1949 |
| `price_usd` asc | 0.1987 |
| `prop_location_score2` desc | 0.2836 |

These set the floor: any learned model has to clearly beat 0.2836 to justify its complexity.